# Gestion de comptes bancaires

Ce TP modélise des comptes bancaires en objets. Une version procédurale, faite de dictionnaires et de fonctions, vous est donnée : lisez-la et exécutez-la, puis réécrivez-la en objets, exercice après exercice. Chaque exercice s'appuie sur le précédent : si vous bloquez, exécutez sa solution avant de passer au suivant. Les montants sont des nombres entiers d'euros.

## Spécifications

- Il existe quatre types de comptes :
  - le compte courant, qui autorise un découvert de 500 € au plus ;
  - le compte joint, un compte courant à deux titulaires ;
  - le compte d'épargne, dont le solde ne peut jamais être négatif ;
  - le livret A, un compte d'épargne dont le solde ne peut pas dépasser son plafond de 22 950 €.
- Tout le monde peut déposer de l'argent sur un compte ; seul l'un de ses titulaires peut en retirer, ou en virer vers un autre compte.
- Un virement n'a lieu que si les deux comptes l'acceptent ; sinon, aucun des deux soldes ne change.
- Un client a au plus un compte de chaque type. Un compte joint compte pour ses deux titulaires.
- Une opération refusée lève une exception qui dit pourquoi.

## La version procédurale

Le code suivant respecte ces règles avec des dictionnaires et des fonctions, sauf la dernière règle sur les clients, qu'il ignore. Exécutez-le, puis la cellule suivante, qui joue un scénario.

In [ ]:
from typing import Any

DECOUVERT_AUTORISE = 500
PLAFOND_LIVRET_A = 22_950


def cree_compte(type_compte: str, *titulaires: str) -> dict[str, Any]:
  return {"type": type_compte, "titulaires": titulaires, "solde": 0}


def verifie_solde(compte: dict[str, Any], nouveau_solde: int) -> None:
  if compte["type"] in ("courant", "joint") and nouveau_solde < -DECOUVERT_AUTORISE:
    raise ValueError(f"Découvert dépassé : le solde passerait à {nouveau_solde} €")
  if compte["type"] in ("épargne", "livret A") and nouveau_solde < 0:
    raise ValueError(f"Solde insuffisant : le solde passerait à {nouveau_solde} €")
  if compte["type"] == "livret A" and nouveau_solde > PLAFOND_LIVRET_A:
    raise ValueError(f"Plafond dépassé : le solde passerait à {nouveau_solde} €")


def verifie_titulaire(compte: dict[str, Any], demandeur: str) -> None:
  if demandeur not in compte["titulaires"]:
    raise ValueError(f"{demandeur} n'est pas titulaire de ce compte")


def deposer(compte: dict[str, Any], montant: int) -> None:
  verifie_solde(compte, compte["solde"] + montant)
  compte["solde"] += montant


def retirer(compte: dict[str, Any], demandeur: str, montant: int) -> None:
  verifie_titulaire(compte, demandeur)
  verifie_solde(compte, compte["solde"] - montant)
  compte["solde"] -= montant


def virer(
  source: dict[str, Any], demandeur: str, cible: dict[str, Any], montant: int
) -> None:
  verifie_titulaire(source, demandeur)
  verifie_solde(source, source["solde"] - montant)
  verifie_solde(cible, cible["solde"] + montant)
  source["solde"] -= montant
  cible["solde"] += montant

In [ ]:
courant = cree_compte("courant", "Isabelle")
epargne = cree_compte("épargne", "Jean")
livret = cree_compte("livret A", "Jean")
joint = cree_compte("joint", "Isabelle", "Jean")
deposer(courant, 1300)
deposer(epargne, 1000)
deposer(livret, 1200)
deposer(joint, 6500)
virer(courant, "Isabelle", joint, 500)
retirer(livret, "Jean", 60)
print(courant["solde"], epargne["solde"], livret["solde"], joint["solde"])

try:
  retirer(epargne, "Jean", 2000)
except ValueError as e:
  print(f"Refusé : {e}")

epargne["solde"] = -1_000_000  # Rien ne l'empêche
print(epargne)

Ce code fonctionne, mais :

- chaque nouveau type de compte ajoute des `if` à `verifie_solde`, et un type mal orthographié (`"Livret A"`) n'y est soumis à aucune règle ;
- rien n'empêche de modifier un solde directement, comme le fait la dernière ligne ;
- toutes les erreurs sont des `ValueError` : l'appelant ne distingue pas un refus prévu par les règles d'une erreur de programmation, ni un découvert dépassé d'un plafond atteint ;
- rien ne relie un client à ses comptes, et la règle d'un compte de chaque type n'est pas appliquée.

Les exercices suivants réécrivent ce code en objets, qui règlent chacun de ces points.

## Les exceptions métier

Écrivez une hiérarchie d'exceptions pour les opérations refusées :

- `BusinessError`, qui hérite d'`Exception`, la classe de base de toutes les autres ;
- `InsufficientFundsError` : le solde deviendrait négatif, ou dépasserait le découvert autorisé ;
- `CeilingExceededError` : le solde dépasserait le plafond du compte ;
- `AccessDeniedError` : le demandeur n'est pas titulaire du compte ;
- `DuplicateAccountError` : le client a déjà un compte de ce type.

Comme les exceptions de la bibliothèque standard, elles portent des noms anglais terminés par `Error`. Vérifiez qu'un `except BusinessError` attrape une `CeilingExceededError`, et affichez son message.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une exception personnalisée est une classe qui hérite d'une autre exception ; son corps peut se limiter à une docstring. Le message est le premier argument passé à l'exception (`raise CeilingExceededError("...")`), et `str(e)` le renvoie. Un `except` attrape l'exception de la classe qu'il nomme et celles de toutes ses sous-classes.

```python
class BusinessError(Exception):
  """Opération refusée par les règles de la banque."""


class InsufficientFundsError(...):
  """..."""


...

try:
  raise CeilingExceededError("Plafond de 22 950 € dépassé")
except BusinessError as e:
  print(f"Refusé : {e}")
```

### Solution

In [ ]:
class BusinessError(Exception):
  """Opération refusée par les règles de la banque."""


class InsufficientFundsError(BusinessError):
  """Le solde deviendrait négatif, ou dépasserait le découvert autorisé."""


class CeilingExceededError(BusinessError):
  """Le solde dépasserait le plafond du compte."""


class AccessDeniedError(BusinessError):
  """Le demandeur n'est pas titulaire du compte."""


class DuplicateAccountError(BusinessError):
  """Le client a déjà un compte de ce type."""


try:
  raise CeilingExceededError("Plafond de 22 950 € dépassé")
except BusinessError as e:
  print(f"Refusé : {type(e).__name__} : {e}")

## La classe abstraite `Compte`

Écrivez une classe abstraite `Compte`, la classe de base de tous les comptes :

- `__init__` reçoit le nom du titulaire, range le n-uplet des titulaires (`("Isabelle",)`) dans l'attribut `titulaires`, et un solde de 0 dans l'attribut `_solde` ;
- la propriété `solde` donne le solde, en lecture seule ;
- la méthode abstraite `_verifie_solde(nouveau_solde)` lève une `BusinessError` si le compte ne peut pas avoir ce solde ;
- `deposer(montant)`, `retirer(demandeur, montant)` et `virer(demandeur, cible, montant)` font les opérations des spécifications, en levant une `AccessDeniedError` quand le demandeur n'est pas titulaire ;
- `__repr__` affiche par exemple `CompteCourant('Isabelle', solde=800)`.

Vérifiez que créer un `Compte` lève une `TypeError` : les opérations se testeront avec les sous-classes de l'exercice suivant.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une propriété sans setter est en lecture seule : `compte.solde = 0` lève une `AttributeError`, alors que les méthodes de la classe modifient `self._solde`. Les opérations vérifient tout avant de modifier quoi que ce soit : `virer` appelle le `_verifie_solde` de la source puis celui de la cible, et ne touche aux soldes qu'ensuite. Chaque sous-classe définira sa propre règle dans `_verifie_solde`, que les opérations de `Compte` appellent sans savoir de quel type de compte il s'agit. Une méthode `_verifie_titulaire(demandeur)` évite d'écrire deux fois la vérification du demandeur. Pour `__repr__`, `", ".join(repr(t) for t in self.titulaires)` donne `'Isabelle', 'Jean'`.

```python
from abc import ABC, abstractmethod


class Compte(ABC):
  def __init__(self, titulaire: str) -> None:
    self.titulaires = (titulaire,)
    self._solde = 0

  @property
  def solde(self) -> int:
    return ...

  @abstractmethod
  def _verifie_solde(self, nouveau_solde: int) -> None: ...

  def _verifie_titulaire(self, demandeur: str) -> None:
    if ...:
      raise AccessDeniedError(f"{demandeur} n'est pas titulaire de ce compte")

  def deposer(self, montant: int) -> None:
    self._verifie_solde(self._solde + montant)
    self._solde += montant

  def retirer(self, demandeur: str, montant: int) -> None: ...

  def virer(self, demandeur: str, cible: "Compte", montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_solde(...)
    cible._verifie_solde(...)
    ...

  def __repr__(self) -> str:
    titulaires = ", ".join(repr(t) for t in self.titulaires)
    return f"{type(self).__name__}({titulaires}, solde={...})"


try:
  Compte("Isabelle")
except TypeError as e:
  print(e)
```

### Solution

In [ ]:
from abc import ABC, abstractmethod


class Compte(ABC):
  def __init__(self, titulaire: str) -> None:
    self.titulaires = (titulaire,)
    self._solde = 0

  @property
  def solde(self) -> int:
    return self._solde

  @abstractmethod
  def _verifie_solde(self, nouveau_solde: int) -> None:
    """Lève une BusinessError si le compte ne peut pas avoir ce solde."""

  def _verifie_titulaire(self, demandeur: str) -> None:
    if demandeur not in self.titulaires:
      raise AccessDeniedError(f"{demandeur} n'est pas titulaire de ce compte")

  def deposer(self, montant: int) -> None:
    self._verifie_solde(self._solde + montant)
    self._solde += montant

  def retirer(self, demandeur: str, montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_solde(self._solde - montant)
    self._solde -= montant

  def virer(self, demandeur: str, cible: "Compte", montant: int) -> None:
    self._verifie_titulaire(demandeur)
    self._verifie_solde(self._solde - montant)
    cible._verifie_solde(cible._solde + montant)
    self._solde -= montant
    cible._solde += montant

  def __repr__(self) -> str:
    titulaires = ", ".join(repr(t) for t in self.titulaires)
    return f"{type(self).__name__}({titulaires}, solde={self._solde})"


try:
  Compte("Isabelle")
except TypeError as e:
  print(f"Refusé : {e}")

## Les types de comptes

Écrivez les quatre types de comptes des spécifications, sous-classes de `Compte` :

- `CompteCourant`, dont l'attribut de classe `decouvert_autorise` vaut 500 ;
- `CompteJoint`, sous-classe de `CompteCourant`, qu'on crée avec ses deux titulaires : `CompteJoint("Isabelle", "Jean")` ;
- `CompteEpargne`, dont le solde ne peut pas être négatif ;
- `LivretA`, sous-classe de `CompteEpargne`, dont l'attribut de classe `plafond` vaut 22 950.

Chacun a aussi un attribut de classe `libelle` (`"compte courant"`, `"compte joint"`…), qui servira à l'exercice suivant. Un solde négatif au-delà du découvert autorisé lève une `InsufficientFundsError`, et un solde au-dessus du plafond une `CeilingExceededError`.

Testez-les ensuite avec la cellule de vérification qui suit la solution.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Chaque classe ne définit que ce qui la distingue : sa règle dans `_verifie_solde`, ses attributs de classe, et, pour `CompteJoint`, un `__init__` qui appelle `super().__init__(titulaire)` puis ajoute le cotitulaire aux `titulaires`. `CompteJoint` hérite de la règle de `CompteCourant`. Le `_verifie_solde` de `LivretA` appelle d'abord celui de `CompteEpargne` avec `super()`, puis vérifie le plafond. Dans une méthode, `self.decouvert_autorise` lit l'attribut de classe.

```python
class CompteCourant(Compte):
  libelle = "compte courant"
  decouvert_autorise = 500

  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < ...:
      raise InsufficientFundsError(
        f"Découvert de {self.decouvert_autorise} € dépassé : "
        f"le solde passerait à {nouveau_solde} €"
      )


class CompteJoint(CompteCourant):
  libelle = "compte joint"

  def __init__(self, titulaire: str, cotitulaire: str) -> None:
    super().__init__(titulaire)
    self.titulaires = ...


class CompteEpargne(Compte):
  ...


class LivretA(CompteEpargne):
  libelle = "livret A"
  plafond = 22_950

  def _verifie_solde(self, nouveau_solde: int) -> None:
    super()._verifie_solde(nouveau_solde)
    ...
```

### Solution

In [ ]:
class CompteCourant(Compte):
  libelle = "compte courant"
  decouvert_autorise = 500

  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < -self.decouvert_autorise:
      raise InsufficientFundsError(
        f"Découvert de {self.decouvert_autorise} € dépassé : "
        f"le solde passerait à {nouveau_solde} €"
      )


class CompteJoint(CompteCourant):
  libelle = "compte joint"

  def __init__(self, titulaire: str, cotitulaire: str) -> None:
    super().__init__(titulaire)
    self.titulaires = (titulaire, cotitulaire)


class CompteEpargne(Compte):
  libelle = "compte d'épargne"

  def _verifie_solde(self, nouveau_solde: int) -> None:
    if nouveau_solde < 0:
      raise InsufficientFundsError(
        f"Solde insuffisant : le solde passerait à {nouveau_solde} €"
      )


class LivretA(CompteEpargne):
  libelle = "livret A"
  plafond = 22_950

  def _verifie_solde(self, nouveau_solde: int) -> None:
    super()._verifie_solde(nouveau_solde)
    if nouveau_solde > self.plafond:
      raise CeilingExceededError(
        f"Plafond de {self.plafond} € dépassé : le solde passerait à {nouveau_solde} €"
      )

### Vérification

La cellule suivante rejoue le scénario de la version procédurale avec vos classes, puis tente des opérations que les règles refusent. La fonction `tente` exécute une opération et affiche si elle est acceptée, ou l'exception métier qui la refuse.

Résultats attendus : des soldes de 800, 1 000, 1 140 et 7 000 € ; la 4e opération acceptée, les autres refusées (`InsufficientFundsError`, `CeilingExceededError`, `AccessDeniedError`, puis `InsufficientFundsError` : le découvert dépasserait 500 €) ; à la fin, -400 € sur le compte courant et toujours 7 000 € sur le compte joint, que le virement refusé n'a pas touché ; et une `AttributeError` quand on affecte le solde.

In [ ]:
from collections.abc import Callable


def tente(description: str, operation: Callable[[], None]) -> None:
  try:
    operation()
  except BusinessError as e:
    print(f"{description} : refusé ({type(e).__name__} : {e})")
  else:
    print(f"{description} : accepté")


courant = CompteCourant("Isabelle")
epargne = CompteEpargne("Jean")
livret = LivretA("Jean")
joint = CompteJoint("Isabelle", "Jean")
courant.deposer(1300)
epargne.deposer(1000)
livret.deposer(1200)
joint.deposer(6500)
courant.virer("Isabelle", joint, 500)
livret.retirer("Jean", 60)
print(courant, epargne, livret, joint, sep="\n")

tente(
  "Jean retire 2000 € de son compte d'épargne", lambda: epargne.retirer("Jean", 2000)
)
tente("Jean dépose 22 000 € sur son livret A", lambda: livret.deposer(22_000))
tente("Jean retire 100 € du compte d'Isabelle", lambda: courant.retirer("Jean", 100))
tente("Isabelle retire 1200 € de son compte", lambda: courant.retirer("Isabelle", 1200))
tente(
  "Isabelle vire 200 € au compte joint", lambda: courant.virer("Isabelle", joint, 200)
)
print(courant, joint, sep="\n")

try:
  courant.solde = 1_000_000
except AttributeError as e:
  print(f"Refusé : {e}")

## Les clients

Écrivez une classe `Client`, créée avec le nom du client, qui associe un client à ses comptes :

- l'attribut `comptes` est un dictionnaire, vide au départ, dont les clés sont les types de comptes (les classes elles-mêmes, comme `CompteCourant`) et les valeurs les comptes du client ;
- `ajouter(compte)` ajoute un compte au client, en levant une `AccessDeniedError` si le client n'en est pas titulaire, et une `DuplicateAccountError` s'il a déjà un compte de ce type ;
- la propriété `patrimoine` donne la somme des soldes de ses comptes ;
- `__repr__` affiche le nom du client et la liste de ses comptes.

Un compte existe sans son client, et un compte joint appartient à deux clients : c'est une agrégation, pas une composition. Testez votre classe avec la cellule de vérification qui suit la solution.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`type(compte)` renvoie la classe d'un compte, qui sert de clé : `CompteJoint` et `CompteCourant` sont deux clés différentes, même si l'une hérite de l'autre. `in` teste si une clé est dans un dictionnaire, et `sum` additionne les soldes que produit une expression génératrice sur `self.comptes.values()`. Le message de `DuplicateAccountError` peut utiliser l'attribut de classe `libelle` du compte : `Isabelle a déjà un compte courant`.

```python
class Client:
  def __init__(self, nom: str) -> None:
    self.nom = nom
    self.comptes: dict[type[Compte], Compte] = {}

  def ajouter(self, compte: Compte) -> None:
    if self.nom not in compte.titulaires:
      raise AccessDeniedError(...)
    if type(compte) in self.comptes:
      raise DuplicateAccountError(f"{self.nom} a déjà un {compte.libelle}")
    self.comptes[...] = compte

  @property
  def patrimoine(self) -> int:
    return sum(...)

  def __repr__(self) -> str:
    return f"Client({self.nom!r}, comptes={list(...)})"
```

### Solution

In [ ]:
class Client:
  def __init__(self, nom: str) -> None:
    self.nom = nom
    self.comptes: dict[type[Compte], Compte] = {}

  def ajouter(self, compte: Compte) -> None:
    if self.nom not in compte.titulaires:
      raise AccessDeniedError(f"{self.nom} n'est pas titulaire de {compte!r}")
    if type(compte) in self.comptes:
      raise DuplicateAccountError(f"{self.nom} a déjà un {compte.libelle}")
    self.comptes[type(compte)] = compte

  @property
  def patrimoine(self) -> int:
    return sum(compte.solde for compte in self.comptes.values())

  def __repr__(self) -> str:
    return f"Client({self.nom!r}, comptes={list(self.comptes.values())})"

### Vérification

Résultats attendus : Isabelle a le compte courant et le compte joint, Jean le compte d'épargne, le livret A et le compte joint ; leurs patrimoines sont de 6 600 et 9 140 € (le compte joint compte pour les deux) ; le second compte courant d'Isabelle est refusé (`DuplicateAccountError`), comme le livret A de Jean (`AccessDeniedError`), mais pas un livret A à son nom.

In [ ]:
isabelle = Client("Isabelle")
jean = Client("Jean")
isabelle.ajouter(courant)
isabelle.ajouter(joint)
jean.ajouter(epargne)
jean.ajouter(livret)
jean.ajouter(joint)
print(isabelle, jean, sep="\n")
print(isabelle.patrimoine, jean.patrimoine)

tente(
  "Isabelle ouvre un second compte courant",
  lambda: isabelle.ajouter(CompteCourant("Isabelle")),
)
tente("Isabelle ajoute le livret A de Jean", lambda: isabelle.ajouter(livret))
tente("Isabelle ouvre un livret A", lambda: isabelle.ajouter(LivretA("Isabelle")))
print(isabelle.comptes[LivretA])